# 06. Memory Architectures: NTM, PKM, PEER, Infini-attention, and Titans

- **Category:** 05_architectures / 03_recurrent_ssm_and_memory
- **Lineage & Citations:** Alex Graves et al., 2014 (*Neural Turing Machines*); Guillaume Lample et al., 2019 (*Product Key Memory*); Tsendsuren Munkhdalai et al., 2024 (*Infini-attention*); Ali Behrouz et al., 2024 (*Titans*).
- **Purpose:** Decouples computational units from explicit associative memory, scaling memory from differentiable RAM matrices (NTM) to constant-footprint compressive states (Infini-attention) and test-time memorization (Titans).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Linear Attention RWKV RetNet GLA DeltaNet and TTT](05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.ipynb) $\to$ `🔗 Downstream Syntheses:` [Reservoir Computing Echo State Networks LSM and NG-RC](07_reservoir_computing_echo_state_networks_lsm_and_ngrc.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> - **Short Contexts (< 2048 tokens):** External memory heads introduce dispatch overhead without providing capacity benefits over standard local attention.
> - **Permanent Knowledge Storage:** In-context neural memory is ephemeral; use vector databases (RAG) or frozen weights for persistent facts.

## Mathematical Foundations & Governing Equations

### 1. NTM Content-Based Addressing
Cosine similarity and normalized read head:
$$
\text{sim}(k, M_i) = \frac{k \cdot M_i}{\|k\|_2 \|M_i\|_2}, \quad w_i = \frac{\exp(\beta \cdot \text{sim}(k, M_i))}{\sum_j \exp(\beta \cdot \text{sim}(k, M_j))}, \quad r = \sum_i w_i M_i
$$

### 2. Infini-attention Compressive Memory
Non-negative kernel retrieval and linear state accumulation:
$$
A_{\text{mem}} = \frac{\phi(Q) M_{s-1}}{\phi(Q) z_{s-1}^\top + \epsilon}, \quad M_s = M_{s-1} + \phi(K)^\top V, \quad z_s = z_{s-1} + \phi(K)
$$

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)

memory = weights["memory"]
print(f"Loaded {len(weights)} golden tensors on {env.device}, NTM memory slots: {list(memory.shape)}")

In [ ]:
def ntm_content_addressing_read(memory, key, beta):
    key_norm = F.normalize(key, p=2, dim=-1).unsqueeze(1)
    mem_norm = F.normalize(memory, p=2, dim=-1)
    cos_sim = (key_norm * mem_norm).sum(dim=-1)
    weights = F.softmax(beta * cos_sim, dim=-1)
    read_vec = torch.bmm(weights.unsqueeze(1), memory).squeeze(1)
    return read_vec, weights

def infini_memory_step(m_prev, z_prev, q, k, v):
    phi_q = F.elu(q) + 1.0
    phi_k = F.elu(k) + 1.0
    num = torch.bmm(phi_q.unsqueeze(1), m_prev).squeeze(1)
    denom = (phi_q * z_prev).sum(dim=-1, keepdim=True) + 1e-4
    a_mem = num / denom
    m_delta = torch.bmm(phi_k.unsqueeze(2), v.unsqueeze(1))
    return a_mem, m_prev + m_delta, z_prev + phi_k

In [ ]:
# Quantitative Mathematical Assertions & Golden Parity Verification
read_vec, ntm_weights = ntm_content_addressing_read(memory, weights["key"], weights["beta"])
err_r = float(torch.max(torch.abs(read_vec - weights["read_vec"])))
err_w = float(torch.max(torch.abs(ntm_weights - weights["ntm_weights"])))
assert err_r < 1e-6, f"NTM read error: {err_r:.2e}"
assert err_w < 1e-6, f"NTM weights error: {err_w:.2e}"

a_mem, m_next, z_next = infini_memory_step(
    weights["m_prev"], weights["z_prev"], weights["q"], weights["k"], weights["v"]
)
err_am = float(torch.max(torch.abs(a_mem - weights["a_mem"])))
err_mn = float(torch.max(torch.abs(m_next - weights["m_next"])))
assert err_am < 1e-6, f"Infini a_mem error: {err_am:.2e}"
assert err_mn < 1e-6, f"Infini m_next error: {err_mn:.2e}"
print(f"Parity Verified: NTM Read={err_r:.2e}, Weights={err_w:.2e}, Infini-Memory={err_am:.2e}")

In [ ]:
import matplotlib.pyplot as plt
from core.benchmark import benchmark_fn, format_benchmark_table

b_ntm = benchmark_fn(lambda: ntm_content_addressing_read(memory, weights["key"], weights["beta"]), iters=50, name="NTM Content Read")
b_inf = benchmark_fn(lambda: infini_memory_step(
    weights["m_prev"], weights["z_prev"], weights["q"], weights["k"], weights["v"]
), iters=50, name="Infini-Memory Step")
print(format_benchmark_table([b_ntm, b_inf]))

# Visualization: NTM Attention Weights & Infini-Memory Matrix
fig, ax = plt.subplots(1, 2, figsize=(8, 2.5), dpi=100)
ax[0].bar(range(ntm_weights.shape[1]), ntm_weights[0].detach().numpy(), color="teal")
ax[0].set_title("NTM Slot Weights w_i")
ax[1].imshow(m_next[0].detach().numpy(), cmap="viridis", aspect="auto")
ax[1].set_title("Infini-Memory State M [D, D]")
plt.tight_layout()
plt.show()

In [ ]:
# Pathological Misuse: Unprotected Zero Denominator in Memory Readout
phi_zero = torch.zeros(1, 4)
z_zero = torch.zeros(1, 4)
num_dummy = torch.ones(1, 4)
# Without epsilon, denom is strictly zero -> NaN!
denom_flawed = (phi_zero * z_zero).sum(dim=-1, keepdim=True)
denom_safe = denom_flawed + 1e-4
out_flawed = num_dummy / denom_flawed
out_safe = num_dummy / denom_safe

print(f"Flawed readout: {out_flawed[0, 0].item()} (is_inf: {torch.isinf(out_flawed).any().item()})")
print(f"Safe readout: {out_safe[0, 0].item():.2e}")
assert torch.isinf(out_flawed).any(), "Zero denominator must produce Inf/NaN without protective epsilon!"
print("Demonstrated requirement of protective epsilon in associative normalizer.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Decoupled Working Memory:** Separating state storage from controller parameters enables complex multi-step reasoning.
2. **Constant-Size Long Context:** Infini-attention processes unlimited context windows with a fixed-size compressive memory matrix.
3. **Online Self-Supervised Updates:** Titans dynamically memorizes test-time surprises via associative gradient updates.

---
### Cross-Domain Generalisation Matrix
- **Robotics Sensory RAM:** Buffers high-rate lidar and vision scans into structured memory slots for spatial navigation.
- **Complex Algorithmic Tasks:** Solves multi-digit arithmetic, sorting, and associative graph lookups.
- **Long-Document Processing:** Continually updates compressive state over megabyte-scale documents.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.md](../../../markdown/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.md)
- **PyTorch 2.x Module:** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.py](../../../pytorch2/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.py)
- **Functional JAX Module:** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.py](../../../jax/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.py)
- **Candle Rust Module:** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.rs](../../../candle/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.rs)
- **Hardware Kernel:** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.rs](../../../kernels/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.rs)
- **Universal Golden SafeTensors:** [06_memory_architectures_ntm_pkm_peer_infini_and_titans.safetensors](../../../weights/05_architectures/03_recurrent_ssm_and_memory/06_memory_architectures_ntm_pkm_peer_infini_and_titans.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous Topic: 05. Linear Attention RWKV RetNet GLA DeltaNet and TTT](05_linear_attention_rwkv_retnet_gla_deltanet_and_ttt.ipynb) | [Category Index](../../../README.md#05_architectures) | [Next Topic: 07. Reservoir Computing Echo State Networks LSM and NG-RC](07_reservoir_computing_echo_state_networks_lsm_and_ngrc.ipynb) &rarr;